# 04 · Chunk Estimation & API Cost Projection

Processes all available PDFs, counts real chunks, and produces a precise cost estimate  
for Claude API usage across the full 32-state compliance pipeline.

**No NLP or scoring in this notebook** — only text extraction, chunk simulation, and arithmetic.

**Data sources:**
- `data/output/legal_inventory.csv` (or `data/legal_inventory.csv`) — 31-state inventory from notebook 01
- `data/morelos/` — Morelos fallback PDFs (local)

> **Note on inventory URLs:** The ordenjuridico.gob.mx inventory links point to HTML viewer pages  
> (`fichaOrdenamiento.php`), not direct PDF download URLs. Attempting to fetch them as PDFs  
> will fail. Those entries are logged as unavailable and the Morelos local PDFs serve as the  
> concrete sample for chunk calibration. States with no local PDFs are extrapolated from the  
> per-document averages derived from the available sample.

## Section 1 · Setup & Imports

In [11]:
%pip install -q pdfplumber pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import json
import pathlib
import warnings

import pandas as pd
import pdfplumber

warnings.filterwarnings("ignore")  # suppress pdfplumber CropBox warnings

NOTEBOOK_DIR = pathlib.Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# ── Chunking parameters (mirrors the real pipeline in notebook 05) ────────────
CHUNK_SIZE    = 1000   # characters
CHUNK_OVERLAP = 200    # characters
CHUNK_STEP    = CHUNK_SIZE - CHUNK_OVERLAP  # effective advance per chunk = 800

print(f"Project root : {PROJECT_ROOT}")
print(f"Chunk size   : {CHUNK_SIZE} chars  |  Overlap: {CHUNK_OVERLAP} chars  |  Step: {CHUNK_STEP} chars")

Project root : C:\Users\deaqu\OneDrive - Universidad Politécnica de Yucatán\Documentos\Carrera Academica\Data Engineer\7_Quadrimester\Internship\Validition-Pipeline-RAG-Law-and-Compliance-LGBTIQ-Rights
Chunk size   : 1000 chars  |  Overlap: 200 chars  |  Step: 800 chars


## Section 2 · Load the Legal Inventory

Reads the CSV exported by notebook 01, then adds the Morelos fallback PDFs.  
Both are consolidated into a single source DataFrame.

**If the CSV has not been exported yet:** uncomment the export cell at the end of  
`01_legal_inventory_exploration.ipynb` and re-run it, then come back here.

In [13]:
# ── Locate inventory CSV (notebook 01 may have saved it to either path) ───────
_candidates = [
    PROJECT_ROOT / "data" / "output" / "legal_inventory.csv",
    PROJECT_ROOT / "data" / "legal_inventory.csv",
]
inventory_path = next((p for p in _candidates if p.exists()), None)

if inventory_path:
    inv_df = pd.read_csv(inventory_path, encoding="utf-8-sig")
    print(f"Inventory loaded from : {inventory_path.relative_to(PROJECT_ROOT)}")
    print(f"  Rows : {len(inv_df)}  |  States : {inv_df['state'].nunique()}")
else:
    print("[WARNING] legal_inventory.csv not found.")
    print("  Run notebook 01 and uncomment the export cell, then re-run this notebook.")
    print("  Proceeding with Morelos fallback PDFs only.")
    inv_df = pd.DataFrame(columns=["state", "law_title", "url", "file_type", "matched_keywords", "law_type"])

# ── Morelos fallback PDFs ─────────────────────────────────────────────────────
MORELOS_DIR = PROJECT_ROOT / "data" / "morelos"
morelos_pdfs = sorted(MORELOS_DIR.glob("*.pdf"))
print(f"\nMorelos fallback PDFs found : {len(morelos_pdfs)}")
for p in morelos_pdfs:
    print(f"  {p.name}")

# ── Build consolidated sources DataFrame ─────────────────────────────────────
rows = []

# Inventory entries: URL-based (not locally downloadable yet)
for _, row in inv_df.iterrows():
    rows.append({
        "state":     row["state"],
        "file_path": row.get("url", ""),
        "file_name": row.get("law_title", ""),
        "source":    "inventory",
    })

# Morelos local PDFs
for p in morelos_pdfs:
    rows.append({
        "state":     "Morelos",
        "file_path": str(p),
        "file_name": p.name,
        "source":    "fallback",
    })

sources_df = pd.DataFrame(rows)
print(f"\nConsolidated sources : {len(sources_df)} entries")
print(sources_df["source"].value_counts().to_string())

Inventory loaded from : data\output\legal_inventory.csv
  Rows : 2639  |  States : 31

Morelos fallback PDFs found : 5
  CONSTMOR.pdf
  CPENALEM.pdf
  LCDERHUMEM.pdf
  LMENTALEM.pdf
  LSALUDEM.pdf

Consolidated sources : 2644 entries
source
inventory    2639
fallback        5


## Section 3 · Text Extraction & Chunk Simulation

For each **local** PDF (fallback source), text is extracted with `pdfplumber` and  
then split into chunks of 1 000 characters with 200-character overlap —  
matching the parameters that will be used in the real pipeline.

Inventory entries (URL-based) cannot be downloaded at this stage because the  
ordenjuridico.gob.mx links are HTML viewer pages, not direct PDF URLs.  
They are logged as unavailable and later handled via extrapolation in Section 4.

**Chunk count formula:**  
`chunk_count = max(1, ceil((char_count - CHUNK_SIZE) / CHUNK_STEP) + 1)`  
for texts longer than one chunk; otherwise 1 chunk.

In [14]:
import math

def simulate_chunks(text: str) -> int:
    """Count LangChain-style chunks for a given text string."""
    n = len(text)
    if n == 0:
        return 0
    if n <= CHUNK_SIZE:
        return 1
    return math.ceil((n - CHUNK_SIZE) / CHUNK_STEP) + 1


extraction_rows = []
unavailable_log = []

for _, row in sources_df.iterrows():
    state     = row["state"]
    file_path = row["file_path"]
    file_name = row["file_name"]
    source    = row["source"]

    # ── URL-based inventory entries — skip, not downloadable yet ──────────────
    if source == "inventory":
        unavailable_log.append({
            "state":     state,
            "file_name": file_name,
            "reason":    "HTML viewer URL — direct PDF not yet extracted",
        })
        continue

    # ── Local PDF ─────────────────────────────────────────────────────────────
    pdf_path = pathlib.Path(file_path)
    if not pdf_path.exists():
        unavailable_log.append({
            "state":     state,
            "file_name": file_name,
            "reason":    "Local file not found",
        })
        continue

    try:
        with pdfplumber.open(pdf_path) as pdf:
            page_count = len(pdf.pages)
            full_text  = "\n".join(page.extract_text() or "" for page in pdf.pages)

        char_count  = len(full_text)
        chunk_count = simulate_chunks(full_text)

        extraction_rows.append({
            "state":       state,
            "file_name":   pdf_path.name,
            "page_count":  page_count,
            "char_count":  char_count,
            "chunk_count": chunk_count,
            "source":      source,
        })
        print(f"  {state:20} {pdf_path.name:30} "
              f"pages={page_count:3}  chars={char_count:7,}  chunks={chunk_count:4}")

    except Exception as exc:
        unavailable_log.append({
            "state":     state,
            "file_name": file_name,
            "reason":    f"Extraction error: {exc}",
        })

extraction_df  = pd.DataFrame(extraction_rows)
unavailable_df = pd.DataFrame(unavailable_log)

print(f"\nExtracted  : {len(extraction_df)} PDFs")
print(f"Unavailable: {len(unavailable_df)} entries")

  Morelos              CONSTMOR.pdf                   pages=386  chars=1,256,565  chunks=1571
  Morelos              CPENALEM.pdf                   pages=355  chars=966,572  chunks=1208
  Morelos              LCDERHUMEM.pdf                 pages= 49  chars=136,977  chunks= 171
  Morelos              LMENTALEM.pdf                  pages= 79  chars=208,611  chunks= 261
  Morelos              LSALUDEM.pdf                   pages=203  chars=520,112  chunks= 650

Extracted  : 5 PDFs
Unavailable: 2639 entries


In [15]:
# ── Unavailable log ───────────────────────────────────────────────────────────
if not unavailable_df.empty:
    print("Unavailable entries:")
    # Group by reason to avoid printing every URL-based entry individually
    reason_counts = unavailable_df["reason"].value_counts()
    for reason, count in reason_counts.items():
        print(f"  [{count:3}]  {reason}")

Unavailable entries:
  [2639]  HTML viewer URL — direct PDF not yet extracted


## Section 4 · Chunk Summary Per State

Aggregates extracted data by state.  

States with no locally available PDFs are **extrapolated** from the Morelos sample  
(per-document averages × number of documents listed in the inventory for that state).  
Extrapolated rows are flagged so they can be distinguished from real measurements.

In [16]:
# ── Aggregate real data ───────────────────────────────────────────────────────
if not extraction_df.empty:
    real_summary = (
        extraction_df
        .groupby("state", as_index=False)
        .agg(
            total_docs   =("file_name",  "count"),
            total_pages  =("page_count", "sum"),
            total_chars  =("char_count", "sum"),
            total_chunks =("chunk_count","sum"),
        )
    )
    real_summary["data_type"] = "real"

    # Per-document averages from the real sample (used for extrapolation)
    avg_pages_per_doc  = extraction_df["page_count"].mean()
    avg_chars_per_doc  = extraction_df["char_count"].mean()
    avg_chunks_per_doc = extraction_df["chunk_count"].mean()
    print(f"Sample averages per document:")
    print(f"  Pages  : {avg_pages_per_doc:.1f}")
    print(f"  Chars  : {avg_chars_per_doc:,.0f}")
    print(f"  Chunks : {avg_chunks_per_doc:.1f}")
else:
    real_summary = pd.DataFrame(columns=["state","total_docs","total_pages","total_chars","total_chunks","data_type"])
    avg_pages_per_doc = avg_chars_per_doc = avg_chunks_per_doc = 0
    print("[WARNING] No extracted PDFs — cannot compute sample averages.")

# ── Extrapolate states that appear only in the inventory ─────────────────────
if not inv_df.empty:
    inventory_state_counts = inv_df.groupby("state")["law_title"].count().reset_index()
    inventory_state_counts.columns = ["state", "doc_count"]

    real_states = set(real_summary["state"]) if not real_summary.empty else set()
    extrap_rows = []
    for _, r in inventory_state_counts.iterrows():
        if r["state"] not in real_states:
            n = r["doc_count"]
            extrap_rows.append({
                "state":        r["state"],
                "total_docs":   n,
                "total_pages":  round(n * avg_pages_per_doc),
                "total_chars":  round(n * avg_chars_per_doc),
                "total_chunks": round(n * avg_chunks_per_doc),
                "data_type":    "extrapolated",
            })
    extrap_summary = pd.DataFrame(extrap_rows)
else:
    extrap_summary = pd.DataFrame()

# ── Combine and display ───────────────────────────────────────────────────────
summary_df = pd.concat([real_summary, extrap_summary], ignore_index=True)
summary_df = summary_df.sort_values("state").reset_index(drop=True)

print(f"\nPer-state summary ({len(summary_df)} states):")
print(summary_df.to_string(index=False))

Sample averages per document:
  Pages  : 214.4
  Chars  : 617,767
  Chunks : 772.2

Per-state summary (32 states):
              state  total_docs  total_pages  total_chars  total_chunks    data_type
     Aguascalientes         193        41379    119229108        149035 extrapolated
    Baja California          14         3002      8648744         10811 extrapolated
Baja California Sur          25         5360     15444185         19305 extrapolated
           Campeche          21         4502     12973115         16216 extrapolated
            Chiapas          36         7718     22239626         27799 extrapolated
          Chihuahua         161        34518     99460551        124324 extrapolated
   Ciudad de Mexico         442        94765    273053191        341312 extrapolated
           Coahuila         258        55315    159383989        199228 extrapolated
             Colima         118        25299     72896553         91120 extrapolated
            Durango          34    

## Section 5 · Cost Estimation

**Assumptions:**

| Parameter | Value | Rationale |
|---|---|---|
| Compliance markers per state | 6 | L1, L2, L3, C1, C2, C3 |
| LangGraph steps per marker (avg) | 3.5 | Devil's advocate loop: retrieve → draft → critique → (optional) revise |
| RAG top-k chunks per step | 5 | Standard retrieval window |
| Tokens per chunk | chars / 4 | Standard approximation (1 token ≈ 4 English chars; Spanish is slightly denser) |
| Extra input tokens per call | 500 | System prompt + marker checklist + instructions |
| Output tokens per call | 400 | Score + justification + cited articles |
| Model | claude-sonnet-4 | Best cost/performance for structured legal analysis |
| Input price | $3.00 / M tokens | claude-sonnet-4 pricing |
| Output price | $15.00 / M tokens | claude-sonnet-4 pricing |

In [17]:
# ── Pricing constants ─────────────────────────────────────────────────────────
MARKERS_PER_STATE      = 6
STEPS_PER_MARKER       = 3.5   # average LangGraph devil's advocate steps
TOP_K                  = 5     # RAG chunks retrieved per step
EXTRA_INPUT_TOKENS     = 500   # system prompt + checklist
OUTPUT_TOKENS_PER_CALL = 400
INPUT_PRICE_PER_M      = 3.00  # USD per million input tokens
OUTPUT_PRICE_PER_M     = 15.00 # USD per million output tokens

LLM_CALLS_PER_STATE = MARKERS_PER_STATE * STEPS_PER_MARKER  # 21.0

cost_rows = []
for _, row in summary_df.iterrows():
    total_chunks     = row["total_chunks"]
    total_chars      = row["total_chars"]

    # Average tokens per chunk for this state
    avg_chunk_tokens = (total_chars / total_chunks / 4) if total_chunks > 0 else 0

    # Tokens per LLM call
    input_tokens_per_call  = (TOP_K * avg_chunk_tokens) + EXTRA_INPUT_TOKENS

    # Total tokens across all calls for this state
    total_input_tokens  = LLM_CALLS_PER_STATE * input_tokens_per_call
    total_output_tokens = LLM_CALLS_PER_STATE * OUTPUT_TOKENS_PER_CALL

    # Cost
    cost_usd = (
        (total_input_tokens  / 1_000_000 * INPUT_PRICE_PER_M) +
        (total_output_tokens / 1_000_000 * OUTPUT_PRICE_PER_M)
    )

    cost_rows.append({
        "state":            row["state"],
        "data_type":        row["data_type"],
        "total_chunks":     int(total_chunks),
        "avg_chunk_tokens": round(avg_chunk_tokens, 1),
        "llm_calls":        LLM_CALLS_PER_STATE,
        "input_tokens":     round(total_input_tokens),
        "output_tokens":    round(total_output_tokens),
        "cost_usd":         round(cost_usd, 4),
    })

cost_df = pd.DataFrame(cost_rows).sort_values("state").reset_index(drop=True)

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_rows", 40)
print(cost_df.to_string(index=False))

              state    data_type  total_chunks  avg_chunk_tokens  llm_calls  input_tokens  output_tokens  cost_usd
     Aguascalientes extrapolated        149035          200.0000    21.0000         31500           8400    0.2205
    Baja California extrapolated         10811          200.0000    21.0000         31500           8400    0.2205
Baja California Sur extrapolated         19305          200.0000    21.0000         31500           8400    0.2205
           Campeche extrapolated         16216          200.0000    21.0000         31501           8400    0.2205
            Chiapas extrapolated         27799          200.0000    21.0000         31500           8400    0.2205
          Chihuahua extrapolated        124324          200.0000    21.0000         31500           8400    0.2205
   Ciudad de Mexico extrapolated        341312          200.0000    21.0000         31500           8400    0.2205
           Coahuila extrapolated        199228          200.0000    21.0000     

## Section 6 · Total Projection

Sums across all 32 states and projects costs for 1, 3, 6, and 12 monthly runs.

In [18]:
total_chunks        = int(cost_df["total_chunks"].sum())
total_llm_calls     = cost_df["llm_calls"].sum()
total_input_tokens  = int(cost_df["input_tokens"].sum())
total_output_tokens = int(cost_df["output_tokens"].sum())
total_cost          = cost_df["cost_usd"].sum()

print("=" * 55)
print("  Full 32-State Pipeline — Single Run")
print("=" * 55)
print(f"  Total chunks          : {total_chunks:>12,}")
print(f"  Total LLM calls       : {total_llm_calls:>12,.0f}")
print(f"  Total input tokens    : {total_input_tokens:>12,}")
print(f"  Total output tokens   : {total_output_tokens:>12,}")
print(f"  Cost per single run   :     ${total_cost:>8.4f} USD")
print("=" * 55)

# ── Monthly run projections ───────────────────────────────────────────────────
projection_rows = []
for runs in [1, 3, 6, 12]:
    projection_rows.append({
        "runs_per_month":     runs,
        "monthly_cost_usd":   round(total_cost * runs, 4),
        "annual_cost_usd":    round(total_cost * runs * 12, 4),
    })

proj_df = pd.DataFrame(projection_rows)
print("\nCost Projection Table:")
print(proj_df.to_string(index=False))

  Full 32-State Pipeline — Single Run
  Total chunks          :    2,041,698
  Total LLM calls       :          672
  Total input tokens    :    1,008,002
  Total output tokens   :      268,800
  Cost per single run   :     $  7.0560 USD

Cost Projection Table:
 runs_per_month  monthly_cost_usd  annual_cost_usd
              1            7.0560          84.6720
              3           21.1680         254.0160
              6           42.3360         508.0320
             12           84.6720       1,016.0640


## Section 7 · Risk Buffer

A **30% buffer** is added to account for:
- Documents longer than the current sample average
- Retries on failed or ambiguous evaluations
- Prompt iterations during development
- Spanish text being token-denser than the `chars / 4` approximation assumes

Both the base estimate and the buffered estimate are reported.

In [19]:
RISK_BUFFER = 0.30

cost_with_buffer = total_cost * (1 + RISK_BUFFER)

print("=" * 55)
print("  Recommended Budget — Single Pipeline Run")
print("=" * 55)
print(f"  Base estimate         :     ${total_cost:>8.4f} USD")
print(f"  +30% risk buffer      :     ${cost_with_buffer:>8.4f} USD")
print("=" * 55)

print("\nBuffered projection by run frequency:")
buffered_rows = []
for runs in [1, 3, 6, 12]:
    buffered_rows.append({
        "runs_per_month":            runs,
        "base_monthly_usd":          round(total_cost * runs, 4),
        "buffered_monthly_usd":      round(cost_with_buffer * runs, 4),
        "buffered_annual_usd":       round(cost_with_buffer * runs * 12, 4),
    })

buf_df = pd.DataFrame(buffered_rows)
print(buf_df.to_string(index=False))

  Recommended Budget — Single Pipeline Run
  Base estimate         :     $  7.0560 USD
  +30% risk buffer      :     $  9.1728 USD

Buffered projection by run frequency:
 runs_per_month  base_monthly_usd  buffered_monthly_usd  buffered_annual_usd
              1            7.0560                9.1728             110.0736
              3           21.1680               27.5184             330.2208
              6           42.3360               55.0368             660.4416
             12           84.6720              110.0736           1,320.8832


## Export

Saves the per-state cost table to `data/output/cost_estimation.csv`.

In [20]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_path = OUTPUT_DIR / "cost_estimation.csv"
cost_df.to_csv(export_path, index=False, encoding="utf-8-sig")
print(f"Exported → {export_path.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(cost_df)}")

Exported → data\output\cost_estimation.csv
Rows: 32
